# **Analisi dei dati di fotoni di Fermi-LAT**
#### A cura di:
#### **P. Cristarella Orestano, S. Cutini, F. de Palma, L. Di Venere, D. Gasparrini**
#### Contatti: [email](mailto:astroparticellebari@gmail.com)

I dati dell'esperimento Fermi-LAT sono memorizzati in file in formato [fits](https://en.wikipedia.org/wiki/FITS) (sia per elenchi che immagini) e facilmente ottenibili dal sito:
https://fermi.gsfc.nasa.gov/ssc/data/access/lat/

In questa analisi useremo google Colab che consente di avere un notebook di [python](https://it.wikipedia.org/wiki/Python) senza bisogno di installare nulla in locale.

Ulteriori informazioni sull'analisi dei flare sono presenti in questo articolo: https://iopscience.iop.org/article/10.1088/0004-637X/722/1/520/meta

#### ISTRUZIONI

Il notebook consente di eseguire comandi in linguaggio python e visualizzare testo, dati e grafici.

Ogni blocco di comandi può essere eseguito cliccando sul tasto **"play"** situato in alto a sinistra della cella, oppure nella cella di codice cliccando **Shift+Invio** o **Ctrl+Invio**.

(All'esecuzione del primo blocco comparirà una finestra di pop-up, nella quale bisogna cliccare sul tasto "Esegui comunque".)

Il notebook può essere modificato, ma le modifiche non verranno salvate.

Per salvare le proprie modifiche è necessario salvare una copia del notebook nel proprio account Google Drive, cliccando in "File" e "Salva una copia su Drive".



---
## Inizializzazione
Iniziamo caricando in Colab delle librerie di Python che ci consentiranno di fare l'analisi dei dati di Fermi-LAT, e definendo alcune funzioni utili in seguito.

In [ ]:
import os,sys
import astropy.io.fits as pyfits
from astropy.wcs import WCS
from astropy.table import Table
import astropy
from astropy import units as u
from astropy.coordinates import SkyCoord, search_around_sky, get_sun
from scipy.optimize import curve_fit
import numpy as np
import time, datetime
from matplotlib.dates import DateFormatter
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
from IPython.display import display, clear_output, Image
from imageio import imread, mimsave

from IPython.utils.text import columnize
from ipywidgets import interactive, interact
import ipywidgets as widgets

import warnings
warnings.simplefilter("ignore")

%matplotlib inline

def secondaryAxis(ax):
  ax_top = ax.twiny()
  ax_top.set_xlim(ax.get_xlim())
  ticks = ax.get_xticks()
  date_labels = []
  for i in range(len(ticks)):
    date_labels.append(met_to_date(ticks[i]).strftime("%d/%m/%Y"))
  ax_top.set_xticklabels(date_labels)
  ax_top.set_xticklabels(date_labels, rotation=20, ha='left')


I tempi di osservazione di Fermi sono espressi in:

Mission Elapsed Time, **MET** (secondi da 01/01/2001 alle ore 00:00)

Definiamo una funzione che per convertirli in data.



In [ ]:
first_time_MET = datetime.datetime(2001,1,1,0,0,0) #tempo di riferimento, primo gennaio 2001

# Definizione funzioni di conversione del tempo:
# met - secondi dall'inizio della missione
# date - data
def met_to_date(met):
  return first_time_MET + met * datetime.timedelta(seconds = 1)

def date_to_met(date):
  return (date - first_time_MET).total_seconds()

# Definizione di funzioni per conversione temporale di array di tempi
def array_datetomet(t):
  tnew = np.zeros(len(t))
  for i in range(len(t)):
    tnew[i] = date_to_met(t[i])
  return tnew
def array_mettodate(t):
  tnew = []
  for i in range(len(t)):
    tnew.append(met_to_date(t[i]))
  return tnew

today = datetime.datetime.now() # oggi
today_met = date_to_met(today)

print("Oggi è il", today.strftime("%d %b %Y"), "che corrisponde a MET = ", int(today_met), " secondi")

---
# Scelta della sorgente

Per oggi abbiamo a disposizione le seguenti sorgenti:

3C279

3C454

4C21

BLLacertae

CTA102

Mrk421

Scegliamo la sorgente scrivendone il nome, tra le virgolette, nella variabile *source_name*.

In [ ]:
# Nome della sorgente
source_name = ""

Carichiamo il file contenente i dati della sorgente.

Leggiamo le coordinate e i tempi di arrivo dei fotoni.

In [ ]:
# CARICAMENTO DATI
url = "https://raw.githubusercontent.com/PaoloCO42/FermiMasterclassPerugia/main/file_" + source_name + ".fits"
data_file = pyfits.open(url)
data_file.info()

times = data_file[1].data["TIME"]
position_RA = data_file[1].data["RA"]   # ASCENSIONE RETTA
position_DEC = data_file[1].data["DEC"] # DECLINAZIONE

position_L = data_file[1].data["L"]     # LONGITUDINE
position_B = data_file[1].data["B"]     # LATITUDINE

energy = data_file[1].data["ENERGY"]

# Check posizione sorgente
positionRA_source = float(data_file[1].header["DSVAL2"].split('(')[-1].split(',')[0])
positionDEC_source = float(data_file[1].header["DSVAL2"].split('(')[-1].split(',')[1])
coordinates_source = SkyCoord(positionRA_source, positionDEC_source, frame='fk5', unit='deg')
galacticLongitude_source = coordinates_source.galactic.l.value
galacticLatitude_source = coordinates_source.galactic.b.value
if max(position_RA)-min(position_RA)>355:
  position_RA = np.where(position_RA>180, position_RA-360, position_RA)
  positionRA_source -= 360
if max(position_L)-min(position_L)>355:
  position_L = np.where(position_L>180, position_L-360, position_L)
  galacticLongitude_source -= 360

data_table = Table(data_file[1].data)


# Inizializzazione parametri per test:
dictionary = {
  "3C279": [5.361e8, 5.367e8, 10, 180, 5.364e8],
  "3C454": [3.1556e8, 3.165e8, 120, 330, 3.1605e8],
  "4C21": [4.3938e8, 4.407e8, 5, 25, 4.398e8],
  "BLLacertae": [6.08e8, 6.11e8, 10,45, 6.093e8],
  "CTA102": [5.052e8, 5.06e8, 100, 500, 5.054e8],
  "Mrk421": [3.63e8, 3.6458e8, 20, 60, 3.64e8]
}
t_start_test = dictionary[source_name][0]
t_stop_test = dictionary[source_name][1]
Fc_test = dictionary[source_name][2]
F0_test = dictionary[source_name][3]
t0_test  = dictionary[source_name][4]


# Troviamo il tempo (in MET) minimo e massimo all'interno del file selezionato
start_met = times.min()
stop_met = times.max()

print('Start: MET ', int(start_met), ' Date ', met_to_date(start_met))
print('Stop: MET ', int(stop_met), ' Date ', met_to_date(stop_met))
print()

# Stampiamo i nomi delle colonne contenute nella tabella dei dati
print("Le colonne nella tabella di dati sono:")
print(columnize(data_table.colnames))

---
# Mappa Conteggi
Osserviamo la sorgente come la vede il Fermi-LAT creando una mappa dei conteggi in ascenzione retta (RA) e declinazione (Dec) che avrà al centro la sorgente e i **10° di cielo** attorno a essa.

In [ ]:
# GRAFICO MAPPA CONTEGGI 10°
fig, ax = plt.subplots(figsize=(10,8), tight_layout = True)
counts, xedges, yedges, im = ax.hist2d(position_RA, position_DEC, bins=50)
cbar=plt.colorbar(im, ax=ax)
cbar.set_label('numero di fotoni', rotation=90)
plt.title(source_name+" - "+"Mappa di conteggi in 10°");
plt.xlabel("Ascensione Retta");
plt.ylabel("Declinazione");
plt.show()

Per analizzarne la curva di luce dobbiamo decidere:
intervallo temporale da analizzare ( **t_start** , **t_stop** )  e campionamento ( **dt** ).

*   **t_start** e **t_stop**, intervallo di tempo da analizzare,
*   **dt**, campionamento.



Scegliamo lo start e lo stop prendendo i limiti dei dati. Più avanti potremo guardare porzioni di dati più brevi.

Impostiamo 1 giorno come campionamento.
Quanti secondi ci sono in un giorno?

Possiamo variare a piacimento intervallo e campionamento
(si consiglia di tenere in conto i start_met e stop_met del file scaricato come limiti, e di non usare campionamenti troppo piccoli).


In [ ]:
# Tempo iniziale, tempo finale, campionamento
t_start = start_met
t_stop = stop_met
dt =

t_array = np.arange(start = t_start, stop = t_stop, step = dt )

Quindi creiamo la curva di luce dei conteggi: il numero di fotoni rivelati dallo strumento in funzione del tempo.

Vengono eliminati i bin temporali dove non vi sono conteggi. Questo accade se il satellite osserva un'altra parte del cielo.


In [ ]:
# GRAFICO CURVA DI LUCE 10°
counts, bin_edges = np.histogram(times,bins = t_array)
bin_centers = 0.5*(bin_edges[1:]+bin_edges[:-1])
times_date = bin_centers

fig, ax = plt.subplots(figsize = (10,6))
plt.xlabel("Tempo MET (s)")
plt.ylabel("Eventi")
plt.title(source_name+" - "+"Curva di luce - conteggi in 10°", fontsize = 16, pad = 20)
plt.errorbar(times_date, counts, yerr = np.sqrt(counts), fmt = "o");
ax.grid(True, color = 'lightgray', linestyle = ':', linewidth = 1, alpha = 1)
secondaryAxis(ax)
plt.show()

La mappa dei conteggi con intorno di 10° dalla sorgente ci fa capire che la curva di luce precedente non rappresenta perfettamente i fotoni ricevuti dalla sorgente.

Quindi per studiare meglio l'emissione dell'AGN dobbiamo fare una selezione più sringente, prendiamo ad esempio solo fotoni entro 1° dall'AGN.

Modichiamo il parametro radius_degree per prendere solo gli eventi molto vicini al centro della nostra sorgente.

In [ ]:
# Scegliamo un raggio (in gradi)
radius_degree =

In [ ]:
# Prendiamo le coordinate della sorgente: coordinates_source, e calcoliamo quelle di tutta la zona
coordinates_all = SkyCoord(position_RA, position_DEC, frame='fk5', unit='deg')

# Calcoliamo la distanza di ogni fotone osservato dal centro della sorgente
sep = coordinates_source.separation(coordinates_all)

index = sep.deg < radius_degree
position_RA_1d = position_RA[index]
position_DEC_1d = position_DEC[index]
times_1d = times[index]

print(f"Riduciamo il numero di eventi da {len(times)} a {len(times_1d)}")

Riguardiamo la mappa dei conteggi centrata sulla sorgente.

L'immagine nei raggi gamma è molto meno "definita" delle immagini a cui siamo abituati ad altre lunghezze d'onda (visibile, radio, X).

Questo è dovuto alla difficoltà di osservare i raggi gamma nello spazio che non ci permette di ottenere elevare risoluzioni angolari (che risulta nella definizione dell'immagine prodotta).

In [ ]:
# GRAFICO MAPPA CONTEGGI INTORNO ALLA SORGENTE
fig, ax = plt.subplots(figsize=(10,8))
counts, xedges, yedges, im = ax.hist2d(position_RA_1d, position_DEC_1d, bins=10)
cbar = plt.colorbar(im, ax=ax)
plt.title(source_name+" - "+f"Mappa di conteggi in {radius_degree}°", fontsize = 16, pad = 10)
plt.xlabel("Ascensione retta")
plt.ylabel("Declinazione")
plt.show()

## Creazione GIF
E' possibile dividere l'intero intervallo temporale in sottointervalli di uguale durata e, per ciascuno di questi sottointervalli, rappresentare una mappa integrata dei fotoni a distanza di un grado dalla sorgente.

In questa maniera avremo altrettante mappe diverse e, mettendole in sequenza creiamo una gif, per ottenere un'immagine di come viene vista la sorgente nel tempo.

È possibile cambiare il numero di intervalli n_intervals per aumentare o diminuire i frame della gif.

In [ ]:
# Numero intervalli per gif
n_intervals =

In [ ]:
# GIF MAPPA CONTEGGI
monthnumbers = ["01","02","03","04","05","06","07","08","09","10","11","12"]
monthnames = ["January","February","March","April","May","June","July","August","September","October","November","December"]
gifimages = []
for i in range(n_intervals):
  gifimages.append("imageforgif"+str(i)+".png")

ra_gif = []
dec_gif = []
for i in range(n_intervals):
  ra_gif.append([])
  dec_gif.append([])

t1 = t_start
t2 = t_stop
date1 = met_to_date(t1)
date2 = met_to_date(t2)

deltatover6 = (t2-t1)/n_intervals
t_gif = []

for i in range(n_intervals+1):
  t_gif.append(t1)
  t1 += deltatover6

for i in range(len(position_RA_1d)):
  for j in range(n_intervals):
    if ((times_1d[i]>t_gif[j]) and (times_1d[i]<t_gif[j+1])):
      ra_gif[j].append(position_RA_1d[i])
      dec_gif[j].append(position_DEC_1d[i])

for i in range(n_intervals):
  firstdate = met_to_date(t_gif[i])
  seconddate = met_to_date(t_gif[i+1])
  d1 = firstdate.strftime("%d")
  m1 = monthnumbers[monthnames.index(firstdate.strftime("%B"))]
  y1 = firstdate.strftime("%Y")
  d2 = seconddate.strftime("%d")
  m2 = monthnumbers[monthnames.index(seconddate.strftime("%B"))]
  y2 = seconddate.strftime("%Y")
  fig, ax = plt.subplots(figsize=(10,8), tight_layout = True)
  counts, xedges, yedges, im =ax.hist2d(ra_gif[i],dec_gif[i],bins=50,vmin=0.,vmax=60./n_intervals)
  cbar=plt.colorbar(im, ax=ax)
  cbar.set_label('Numero di fotoni', rotation=90)
  if y1 == y2:
    plt.title(d1+"/"+m1+"    -    "+d2+"/"+m2+"/"+y2, fontsize = 18, pad = 15)
  else:
    plt.title(d1+"/"+m1+"/"+y1+"    -    "+d2+"/"+m2+"/"+y2, fontsize = 18, pad = 15)

  plt.xlim(positionRA_source-0.8, positionRA_source+0.8)
  plt.ylim(positionDEC_source-0.8, positionDEC_source+0.8)
  plt.xlabel("Ascensione retta (°)")
  plt.ylabel("Declinazione (°)")
  fig.savefig(gifimages[i])
  plt.close(fig)

from PIL import Image

frames = []

for image in gifimages:
  x = Image.open(image)
  frames.append(x)
frame_one = frames[0]
frame_one.save("Source_gif.gif", format="GIF", append_images=frames, save_all=True, duration=1000, loop=3)

from IPython.display import Image as Img

with open('./Source_gif.gif','rb') as f:
    display(Img(data=f.read(), format='png'))

---
# Curva di Luce della sorgente
Ora creiamo nuovamente la curva di luce dei conteggi, più rappresentativa dell'emissione della sorgente.

Quanto è diversa dalla curva vista precedentemente? Perché?

In [ ]:
# GRAFICO CURVA DI LUCE SORGENTE
counts, bin_edges = np.histogram(times_1d, bins = t_array)
bin_centers = 0.5*(bin_edges[1:]+bin_edges[:-1])
times_date = bin_centers

fig, ax = plt.subplots(figsize=(10,6))
plt.xlabel("Tempo MET (s)")
plt.ylabel("Eventi")
plt.title(source_name+" - "+f"Curva di luce - conteggi in {radius_degree}°", fontsize = 16, pad = 20)
plt.errorbar(times_date, counts, yerr = np.sqrt(counts), fmt = "o");
ax.grid(True, color = 'lightgray', linestyle = ':', linewidth = 1, alpha = 1)
secondaryAxis(ax)
plt.show()

---
# **Analisi Dati**

Nelle analisi scientifiche si usa il flusso e non i conteggi perchè il secondo non dipende dalle caratteristiche dell'osservazione e può essere confrontato tra diversi esperimenti.

Per ottenere il flusso dobbiamo dividere il numero di conteggi per l'esposizione.

L'esposizione è analoga al tempo di esposizione di una fotografia: maggiore è il tempo di esposizione maggiore il numero di fotoni rivelati a parità di flusso.

Per oggi, come analisi preliminare, possiamo accontentarci dei conteggi.

---
## **Analisi di un flare**

Scegliamo un flare da analizzare.

Decidiamo approssimativamente l'inizio e la fine del flare (t_start_flare, t_stop_flare) in MET.

Possiamo fare un taglio dei dati per vedere meglio il flare, mettere ***cut_data = True***.

In [ ]:
# Tempo (MET) di inizio e di fine del Flare scelto
t_start_flare = t_start_test
t_stop_flare = t_stop_test

cut_data = False

In [ ]:
# GRAFICO CURVA DI LUCE SINGOLO FLARE
counts, bin_edges = np.histogram(times_1d, bins = t_array)
bin_centers = 0.5*(bin_edges[1:]+bin_edges[:-1])
times_date = bin_centers

fig, ax = plt.subplots(figsize=(10,6))
plt.xlabel("Tempo MET (s)")
plt.ylabel("Eventi")
plt.errorbar(times_date, counts, yerr = np.sqrt(counts),fmt="o");
ax.grid(visible = True, which = 'major', color='lightgray', linestyle=':', linewidth=1, alpha=1)
if cut_data:
  ax.xaxis.set_minor_locator(AutoMinorLocator())
  ax.yaxis.set_minor_locator(AutoMinorLocator())
  ax.grid(which='minor', linestyle='-', linewidth=0.5, alpha = 0.1)
  index = (times_date > t_start_flare) & (times_date < t_stop_flare)
  ax.set_xlim(t_start_flare, t_stop_flare)
  ax.set_ylim(top = max(counts[index])*1.3)
  plt.title(source_name+" - "+f"Singolo Flare - conteggi in {radius_degree}°", fontsize = 16, pad = 20)
else:
  plt.axvspan(t_start_flare, t_stop_flare, color='tab:red', alpha=0.2)
  plt.title(source_name+" - "+f"Curva di luce - conteggi in {radius_degree}°", fontsize = 16, pad = 20)
secondaryAxis(ax)
plt.show()



### Fit Flare
Scelto il Flare  da analizzare dobbiamo definire una funzione per fare un fit ed estrapolare le caratteristiche del flare.

La funzione per il fit è:

#### $F(t) = F_c + F_0 \left( \exp{ \left( \frac{t_0 - t}{ T_r} \right)}  + \exp{ \left( \frac{t - t_0}{ T_d}\right)}  \right)^{-1}$

Con:
*   $F_c$ conteggi del fondo,
*   $F_0$ conteggio massimo del picco,
*   $t_0$ tempo corrispondente al massimo del picco,
*   $T_r$ e $T_d$ tempo di *rise* (salita) e *decay* (discesa), distanza da $t_0$ corrispondente a una diminuzione del flusso dell'80%.





In [ ]:
def FlareFunction(t, Fc, F0, t0, Tr, Td):
  return Fc + F0 / ( np.exp((t0-t)/Tr)  + np.exp((t-t0)/Td) )

In [ ]:
# Stima iniziale dei parametri
Fc = Fc_test
F0 = F0_test
t0 = t0_test
Tr = 3 * 86400     # corrispondente a 3 giorni, se il fit fallisce può dover essere cambiato
Td = 3 * 86400     # corrispondente a 3 giorni, se il fit fallisce può dover essere cambiato

In [ ]:
# GRAFICO FLARE CON FIT
counts, bin_edges = np.histogram(times_1d, bins = t_array)
bin_centers = 0.5*(bin_edges[1:]+bin_edges[:-1])
times_date = bin_centers

maskData = (times_date > t_start_flare) & (times_date < t_stop_flare)
x = times_date[maskData]
y = counts[maskData]
yerr = np.sqrt(counts[maskData])

parameters, covmat = curve_fit(
    FlareFunction,
    xdata = x, ydata = y,
    p0 = [Fc, F0, t0, Tr, Td]
)
parameters_uncertainty = np.sqrt(np.diag(covmat))

# Calcolo Chi quadro ridotto
residuals = (y - FlareFunction(x, *parameters))/yerr
chi2 = np.sum((residuals)**2)
dof = len(x) - len(parameters)
chi2_red = chi2 / dof


fig, ax = plt.subplots(figsize=(10,6))
plt.xlabel("Tempo MET (s)")
plt.ylabel("Eventi")
x_plot = np.linspace(x.min(), x.max(), 1000)
ax.plot(x_plot, FlareFunction(x_plot, *parameters), ls = "solid", color = "tab:red");
plt.errorbar(times_date, counts, yerr = np.sqrt(counts),fmt="o");
ax.grid(visible = True, which = 'major', color='lightgray', linestyle=':', linewidth=1, alpha=1)
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.grid(which='minor', linestyle='-', linewidth=0.5, alpha = 0.1)
index = (times_date > t_start_flare) & (times_date < t_stop_flare)
ax.set_xlim(t_start_flare, t_stop_flare)
ax.set_ylim(top = max(counts[index])*1.3)
plt.title(source_name+" - "+f"Singolo Flare - conteggi in {radius_degree}°", fontsize = 16, pad = 20)
textstr = (
    #f"Fc = {parameters[0]:.0f} ± {parameters_uncertainty[0]:.0f}\n"
    #f"F0 = {parameters[1]:.0f} ± {parameters_uncertainty[1]:.0f}\n"
    #f"t0 = {parameters[2]:.2E} ± {parameters_uncertainty[2]:.2E}\n"
    f"Tr = {parameters[3]:.2E} ± {parameters_uncertainty[3]:.2E}\n"
    f"Td = {parameters[4]:.2E} ± {parameters_uncertainty[4]:.2E}\n"
    f"$\\chi ^2 /dof = {chi2_red:.2f}$"
)
ax.text(0.02, 0.98, textstr, transform = ax.transAxes, fontsize = 10, verticalalignment = 'top', horizontalalignment = 'left', linespacing = 1.6)
secondaryAxis(ax)
plt.show()

---
# Risultati
Dal fit del flare estrapoliamo la durata del Flare, $T_F$, e la simmetria $\xi$:

## $T_F = 2(T_r + T_d)$

## $\xi = \frac{T_d - T_r}{T_d + T_r} $



In [ ]:
Tf = 2*(parameters[3]+parameters[4])

xi = (parameters[4]-parameters[3])/(parameters[4]+parameters[3])

print(f"Durata del flare: {int(Tf)} seconds ({round(Tf/86400, 1)} days)\n")
print(f"Simmetria del flare: {round(xi,2)}")

Cercare altri flare nella curva di luce e trovare e salvare tempo del picco, durata e simmetria:

In [ ]:
# Risultati Flare della sorgente
"""
Flare 1:
t0 =
Tf =
xi =

Flare 2:
t0 =
Tf =
xi =

Flare 3:
t0 =
Tf =
xi =

...
"""

---
# Conclusioni

I Blazar osservati dal Fermi-LAT nei raggi gamma mostrano due tipologie di profilo temporale:
* Fondo stabile e attività sporadiche di flare.

* Complesso e strutturato, forte attività e andamenti a bassissime frequenze.

La morfologia dei flare è legata alle caratteristiche di emissione della sorgente, quindi può essere usata per identificare la classe della sorgente:

* Variabilità con flare piccoli e brevi è spiegabile con disturbi e inomogeneità del processo di accrescimento.

* L'intermittenza può evidenziare la dissipazione nel jet ed è spiegabile con processi guidati dalla turbolenza.


* Profili **Asimmetrici** dei flare sono dovuti a veloci iniezioni di particelle e lento raffreddamento radiativo (cooling-dominated) oppure inverse compton con fotoni prodotti fuori dal jet.

* Profili **Simmetrici** dei flare sono legati al tempo di attraversamento di radiazione attraverso la zona di emissione, dominato dalla geometria, che risulta maggiore del tempo di accrescimento e di raffreddamento. Anche la sovrapposizione di più episodi brevi può produrre flare simmetrici.


